In [27]:
import pandas as pd
import numpy as np
import os

MODEL_OUTPUT = r'..\..\..\outputs2020'
MODEL_INPUT = r'..\..\..\inputs2020'
MODEL_OUTPUT_2010 = r'C:\projects\IdahoSTDM\ITD_STDM\Models\BaseYear\Development\IdahoSTDM\ITDSTDM\outputs_2010'
MODEL_INPUT_2010 = r'C:\projects\IdahoSTDM\ITD_STDM\Models\BaseYear\Development\IdahoSTDM\ITDSTDM\inputs'

sdt_trips = pd.read_csv(os.path.join(MODEL_OUTPUT, 'SDTPersonTrips.csv'))
sdt_trips_2010 = pd.read_csv(os.path.join(MODEL_OUTPUT_2010, 'SDTPersonTrips.csv'))

taz = pd.read_csv(os.path.join(MODEL_INPUT, 'tazs.csv'))
taz_2010 = pd.read_csv(os.path.join(MODEL_INPUT_2010, 'tazs.csv'))
person_data = pd.read_csv(os.path.join(MODEL_OUTPUT, 'PersonData.csv')).merge(taz[['STDM_TAZ', 'State', 'County']], how = 'left', left_on = 'home_taz', right_on = 'STDM_TAZ')
person_data_2010 = pd.read_csv(os.path.join(MODEL_OUTPUT_2010, 'PersonData.csv')).merge(taz_2010[['STDM_TAZ', 'State', 'County']], how = 'left', left_on = 'home_taz', right_on = 'STDM_TAZ')



In [ ]:
sdt_trips['Year'] = 2020
sdt_trips_2010['Year'] = 2010
person_data['Year'] = 2020
person_data_2010['Year'] = 2010
taz['Year'] = 2020
taz_2010['Year'] = 2010


sdt_trips_all = pd.concat([sdt_trips, sdt_trips_2010], axis=0)

person_data_all = pd.concat([person_data, person_data_2010], axis=0)




In [32]:
summary_data = pd.DataFrame({
    'data': ['persons', 'workers', 'employment'],
    'model': [person_data.shape[0], person_data[person_data['ESR'] > 0].shape[0], taz['TotEmp'].sum()],
    'Idaho': [person_data[person_data['State'] == 'Idaho'].shape[0], person_data[(person_data['ESR'] > 0) & (person_data['State'] == 'Idaho')].shape[0], taz[taz['State'] == 'Idaho']['TotEmp'].sum()],
    'model2010': [person_data_2010.shape[0], person_data_2010[person_data['ESR'] > 0].shape[0], taz_2010['TotEmp'].sum()],
    'Idaho2010': [person_data_2010[person_data_2010['State'] == 'Idaho'].shape[0], person_data_2010[(person_data_2010['ESR'] > 0) & (person_data_2010['State'] == 'Idaho')].shape[0], taz_2010[taz_2010['State'] == 'Idaho']['TotEmp'].sum()]

})
summary_data.style.format({'model': '{:,.0f}', 
                           'Idaho': '{:,.0f}',
                           'model2010': '{:,.0f}', 
                           'Idaho2010': '{:,.0f}'})

C:\Users\aditya.gore\AppData\Local\Temp\6\ipykernel_19340\1284518237.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  'model2010': [person_data_2010.shape[0], person_data_2010[person_data['ESR'] > 0].shape[0], taz_2010['TotEmp'].sum()],


,data,model,Idaho,model2010,Idaho2010
0,persons,"3,004,078","1,717,984","2,683,207","1,546,111"
1,workers,"1,775,435","1,001,122","1,580,941","690,298"
2,employment,"1,536,538","945,609","1,100,898","617,369"


In [88]:
temp_df = person_data_all.groupby(['Year', 'State']).size().reset_index(name='Records').pivot_table(index='State', columns='Year', values='Records', fill_value=0)

temp_total = pd.DataFrame(temp_df.sum(numeric_only=True)).T
multi_index = ['Total']
# multi_index = pd.MultiIndex.from_tuples([('Total', 'All')], names=['State', 'tourMode'])
temp_total.index = multi_index
temp_df = pd.concat([temp_df, temp_total])
temp_df['diff'] = temp_df[2020]-temp_df[2010]
temp_df['pct_diff'] = (temp_df['diff']/temp_df[2010])*100


temp_df.style.format({
    2010: '{:,.0f}',  # Add commas to VMT values
    2020: '{:,.0f}',
    'diff': '{:,.0f}',
    'pct_diff': '{:.2f}%'  # Format percentage with two decimal places
})

Year,2010,2020,diff,pct_diff
Idaho,"1,546,111","1,717,984","171,873",11.12%
Montana,"282,454","392,629","110,175",39.01%
Nevada,"61,338","86,443","25,105",40.93%
Oregon,"50,217","62,704","12,487",24.87%
Utah,"157,166","202,495","45,329",28.84%
Washington,"547,668","494,923","-52,745",-9.63%
Wyoming,"38,253","46,900","8,647",22.60%
Total,"2,683,207","3,004,078","320,871",11.96%


In [89]:
temp_df = person_data_all.groupby(['Year', 'State', 'ESR']).size().reset_index(name='Records').pivot_table(index=['State','ESR'], columns='Year', values='Records', fill_value=0)

temp_total = pd.DataFrame(temp_df.sum(numeric_only=True)).T
# multi_index = ['Total']
multi_index = pd.MultiIndex.from_tuples([('Total', 'All')], names=['State', 'ESR'])
temp_total.index = multi_index
temp_df = pd.concat([temp_df, temp_total])
temp_df['diff'] = temp_df[2020]-temp_df[2010]
temp_df['pct_diff'] = (temp_df['diff']/temp_df[2010])*100


temp_df.style.format({
    2010: '{:,.0f}',  # Add commas to VMT values
    2020: '{:,.0f}',
    'diff': '{:,.0f}',
    'pct_diff': '{:.2f}%'  # Format percentage with two decimal places
})

In [39]:
sdt_trips_all.groupby('Year').size().map(lambda x: f'{x:,.0f}')


Year
2010     9,100,835
2020    10,188,002
dtype: object

In [48]:
sdt_trips_all.groupby(['Year', 'tourMode']).size().reset_index(name='Records').pivot_table(index='tourMode', columns='Year', values='Records', fill_value=0).map(lambda x: f'{x:,.0f}')

Year,2010,2020
tourMode,,
AutoDriver,"5,950,718","6,965,881"
AutoPassenger,"2,733,294","2,829,704"
Bike,"125,908","123,539"
Walk,"290,915","268,878"


In [43]:
sdt_trips_all.groupby('Year').distance.describe().map('{:,.2f}'.format)

,count,mean,std,min,25%,50%,75%,max
Year,,,,,,,,
2010,"9,100,835.00",12.68,15.80,0.00,2.22,6.39,17.82,439.45
2020,"10,188,002.00",20.77,29.03,0.00,2.67,8.17,31.32,616.80


In [18]:
sdt_trips.head()

,hhID,memberID,weekdayTour(yes/no),tour#,subTour(yes/no),tourPurpose,tourSegment,tourMode,origin,destination,time,distance,tripStartTime,tripEndTime,tripPurpose,tripMode,income,age,enroll,esr
0,667419,1782636,1,0,0,WORK,0,AutoPassenger,2754,2754,85,72.39,500,500,WORK,SR2,4596,23,3,1
1,667419,1782636,1,0,0,WORK,1,AutoPassenger,2754,2754,85,72.39,800,800,HOME,SR2,4596,23,3,1
2,667419,1782636,1,1,0,WORK,1,AutoPassenger,2754,2754,85,72.39,1500,1500,WORK,SR2,4596,23,3,1
3,667419,1782636,1,1,0,WORK,2,AutoPassenger,2754,2754,85,72.39,2200,2200,HOME,SR2,4596,23,3,1
4,667420,1782637,1,0,0,WORK,0,AutoPassenger,2754,2754,85,72.39,500,500,WORK,SR2,4596,23,3,1


In [86]:
temp_df = sdt_trips_all[~sdt_trips_all.tourMode.isin(['Bike', 'Walk'])].merge(person_data_all[['Year','HH_ID', 'memberID', 'State']], how='left', left_on=['Year','hhID', 'memberID'], right_on=['Year','HH_ID', 'memberID']).\
    groupby(['Year','State','tourMode']).size().reset_index(name='Records').\
        pivot_table(columns='Year', index=['State','tourMode'], values='Records', fill_value=0)

temp_total = pd.DataFrame(temp_df.sum(numeric_only=True)).T
multi_index = pd.MultiIndex.from_tuples([('Total', 'All')], names=['State', 'tourMode'])
temp_total.index = multi_index
temp_df = pd.concat([temp_df, temp_total])
temp_df['diff'] = temp_df[2020]-temp_df[2010]
temp_df['pct_diff'] = (temp_df['diff']/temp_df[2010])*100


temp_df.style.format({
    2010: '{:,.0f}',  # Add commas to VMT values
    2020: '{:,.0f}',
    'diff': '{:,.0f}',
    'pct_diff': '{:.2f}%'  # Format percentage with two decimal places
})

In [51]:
sdt_trips_all.merge(person_data_all[['Year','HH_ID', 'memberID', 'State']], how='left', left_on=['Year','hhID', 'memberID'], right_on=['Year','HH_ID', 'memberID']).groupby(['Year','State','tourMode']).distance.sum().reset_index(name='Records').pivot_table(columns='Year', index=['State','tourMode'], values='Records', fill_value=0).map(lambda x: f'{x:,.0f}')

Year                            2010        2020
State      tourMode                             
Idaho      AutoDriver     35,618,898  47,717,385
           AutoPassenger  14,000,476  15,339,546
           Bike              173,517     175,876
           Walk              167,074     168,079
Montana    AutoDriver     14,107,739  49,776,485
           AutoPassenger   5,282,483  19,291,042
           Bike                  345          10
           Walk                   13           2
Nevada     AutoDriver      6,032,102  16,947,398
           AutoPassenger   2,436,112   7,218,464
           Bike                    0           4
           Walk                    5          45
Oregon     AutoDriver      4,380,662   8,002,424
           AutoPassenger   1,717,220   2,680,443
           Bike                    6          30
           Walk                   14          34
Utah       AutoDriver      6,647,830  14,464,750
           AutoPassenger   4,604,471   5,962,769
           Bike                  125          44
           Walk                    2         140
Washington AutoDriver     10,745,046  13,729,013
           AutoPassenger   5,710,877   5,853,073
           Bike               58,911      44,071
           Walk               68,481      46,912
Wyoming    AutoDriver      2,399,080   3,034,542
           AutoPassenger   1,218,494   1,197,093

In [53]:
person_data_all[['Year','HH_ID', 'memberID', 'State']].groupby(['Year','State']).size().reset_index(name='Records').pivot_table(columns='Year', index=['State'], values='Records', fill_value=0).map(lambda x: f'{x:,.0f}')

Year,2010,2020
State,,
Idaho,"1,546,111","1,717,984"
Montana,"282,454","392,629"
Nevada,"61,338","86,443"
Oregon,"50,217","62,704"
Utah,"157,166","202,495"
Washington,"547,668","494,923"
Wyoming,"38,253","46,900"


In [9]:
print(f"{sdt_trips.distance.sum():,.0f}")

211,649,674


In [10]:
sdt_trips[sdt_trips['tourMode'] == 'AutoDriver'].distance.describe().map('{:,.2f}'.format)

count    6,965,881.00
mean            22.06
std             31.30
min              0.00
25%              2.95
50%              8.84
75%             31.32
max            616.80
Name: distance, dtype: object

In [11]:
print(f"{sdt_trips[sdt_trips['tourMode'] == 'AutoDriver'].distance.sum():,.0f}")


153,671,995
